### 🔑 Step 1: Load your vocabs


In [4]:
import json

with open("../Encodings/vocab_Urdu.json", "r", encoding="utf-8") as f:
    vocab_urdu = json.load(f)

with open("../Encodings/vocab_Roman.json", "r", encoding="utf-8") as f:
    vocab_roman = json.load(f)

# also build reverse maps if needed
id2urdu = {i: t for t, i in vocab_urdu.items()}
id2roman = {i: t for t, i in vocab_roman.items()}


### 🔑 Step 2: Load your dataset

In [ ]:
with open("../CombinedData/src_.txt", encoding="utf-8") as f:
    urdu_lines = [ln.strip() for ln in f if ln.strip()]

with open("../CombinedData/tgt_normalized.txt", encoding="utf-8") as f:
    roman_lines = [ln.strip() for ln in f if ln.strip()]

assert len(urdu_lines) == len(roman_lines), "Mismatch between Urdu and Roman lines!"


In [6]:
def encode_sentence(sentence, token2id, is_urdu=True):
    """
    Encode a sentence using greedy longest-match subword tokenization.
    - `token2id` is your BPE subword vocab (already contains things like "mohabbat_").
    - For Urdu: prepend "_" to each word.
    - For Roman: append "_" to each word.
    """
    tokens = []

    if is_urdu:
        words = ["_" + w for w in sentence.split()]   # Urdu side
    else:
        words = [w + "_" for w in sentence.split()]   # Roman side

    for w in words:
        i = 0
        while i < len(w):
            # try to find the longest subword starting at position i
            subword = None
            for j in range(len(w), i, -1):
                piece = w[i:j]
                if piece in token2id:
                    subword = piece
                    break
            if subword is None:
                tokens.append(token2id["<unk>"])
                i += 1
            else:
                tokens.append(token2id[subword])
                i += len(subword)

    # add special tokens
    return [token2id["<sos>"]] + tokens + [token2id["<eos>"]]


In [7]:
# ---------- Encode dataset ----------
src_ids, tgt_ids = [], []

for urdu, roman in zip(urdu_lines, roman_lines):
    s = encode_sentence(urdu, vocab_urdu, is_urdu=True)
    t = encode_sentence(roman, vocab_roman, is_urdu=False)
    src_ids.append(s)
    tgt_ids.append(t)



In [8]:
src_ids[:3]

[[4, 18, 159, 116, 383, 234, 316, 114, 159, 24, 364, 383, 84, 296, 2],
 [4, 239, 268, 289, 343, 326, 299, 365, 326, 299, 84, 296, 2],
 [4, 24, 364, 444, 203, 352, 462, 234, 316, 100, 435, 364, 189, 159, 36, 2]]

In [9]:
tgt_ids[:2]

[[15, 28, 409, 133, 333, 194, 125, 409, 472, 57, 225, 137, 13],
 [15, 481, 244, 271, 178, 170, 56, 432, 18, 178, 171, 225, 137, 13]]

In [10]:
import json
import torch


In [11]:

# Save as PyTorch tensors
torch.save({
    "src_ids": src_ids,
    "tgt_ids": tgt_ids
}, "../Encodings/encoded_dataset.pt")

print("Saved encoded dataset to Encodings/encoded_dataset.pt")

# Optionally also save as JSON (easy to inspect/debug)
with open("../Encodings/encoded_dataset.json", "w", encoding="utf-8") as f:
    json.dump({
        "src_ids": src_ids,
        "tgt_ids": tgt_ids
    }, f, ensure_ascii=False, indent=2)

print("Preview saved to Encodings/encoded_dataset.json")


Saved encoded dataset to Encodings/encoded_dataset.pt
Preview saved to Encodings/encoded_dataset.json


In [12]:
import torch
import torch.nn as nn


In [13]:
# embedding dimension
embed_dim = 512  

In [14]:
# vocab sizes
src_vocab_size = len(vocab_urdu)
tgt_vocab_size = len(vocab_roman)
print(f"Source vocab size: {src_vocab_size}, Target vocab size: {tgt_vocab_size}")


Source vocab size: 512, Target vocab size: 512


In [15]:
# define embedding layers
src_embedding = nn.Embedding(src_vocab_size, embed_dim, padding_idx=vocab_urdu["<pad>"])
tgt_embedding = nn.Embedding(tgt_vocab_size, embed_dim, padding_idx=vocab_roman["<pad>"])


In [16]:
# -------- Padding function --------
def pad_sequences(seqs, pad_id):
    max_len = max(len(s) for s in seqs)
    tensor = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    lengths = []
    for i, s in enumerate(seqs):
        tensor[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        lengths.append(len(s))
    return tensor, torch.tensor(lengths)


In [17]:
# -------- Pad entire dataset --------
src_tensor, src_lengths = pad_sequences(src_ids, vocab_urdu["<pad>"])
tgt_tensor, tgt_lengths = pad_sequences(tgt_ids, vocab_roman["<pad>"])

print("Full src padded shape:", src_tensor.shape)   # (num_sentences, max_src_len)
print("Full tgt padded shape:", tgt_tensor.shape)   # (num_sentences, max_tgt_len)


Full src padded shape: torch.Size([20856, 41])
Full tgt padded shape: torch.Size([20856, 48])


In [18]:

# -------- Apply embeddings --------
embedded_src = src_embedding(src_tensor)  # (num_sentences, max_src_len, embed_dim)
embedded_tgt = tgt_embedding(tgt_tensor)  # (num_sentences, max_tgt_len, embed_dim)

print("Embedded src shape:", embedded_src.shape)
print("Embedded tgt shape:", embedded_tgt.shape)

Embedded src shape: torch.Size([20856, 41, 512])
Embedded tgt shape: torch.Size([20856, 48, 512])


In [ ]:

# Save padded integer tensors (IDs)
torch.save({
    "src_tensor": src_tensor,      # shape (20856, 41)
    "src_lengths": src_lengths,    # lengths of each src sentence
    "tgt_tensor": tgt_tensor,      # shape (20856, 48)
    "tgt_lengths": tgt_lengths,    # lengths of each tgt sentence
    "vocab_src": vocab_urdu,
    "vocab_tgt": vocab_roman
}, "../TensorData/padded_dataset.pt")

print("Saved padded dataset to TensorData/padded_dataset.pt")

# Save embedded tensors (dense vectors)
torch.save({
    "embedded_src": embedded_src,  # shape (20856, 41, embed_dim)
    "embedded_tgt": embedded_tgt   # shape (20856, 48, embed_dim)
}, "../Embeddings/embedded_dataset.pt")

print("Saved embedded dataset to Embeddings/embedded_dataset.pt")


Saved padded dataset to TensorData/padded_dataset.pt
Saved embedded dataset to Embeddings/embedded_dataset.pt


In [21]:
import json

# Save a small sample (first 3 sentences) for inspection
sample = {
    "src_tensor": src_tensor[:3].tolist(),
    "src_lengths": src_lengths[:3].tolist(),
    "tgt_tensor": tgt_tensor[:3].tolist(),
    "tgt_lengths": tgt_lengths[:3].tolist(),
    "vocab_src": vocab_urdu,
    "vocab_tgt": vocab_roman
}

with open("../TensorData/padded_sample.json", "w", encoding="utf-8") as f:
    json.dump(sample, f, ensure_ascii=False, indent=2)

print("Saved sample to TensorData/padded_sample.json")


Saved sample to TensorData/padded_sample.json


In [ ]:
sample = {
    "embedded_src": embedded_src[:2].tolist(),  # shape (2, 41, embed_dim)
    "embedded_tgt": embedded_tgt[:2].tolist()   # shape (2, 48, embed_dim)
}

with open("../Embeddings/embedded_dataset.json", "w", encoding="utf-8") as f:
    json.dump(sample, f, ensure_ascii=False, indent=2)

print("Saved sample to Embeddings/embedded_dataset.json")

Saved sample to Embeddings/embedded_dataset.json
